# Copilot Agent Transcript — Direct Ingester + Parser (Fabric)

**Ingests** Copilot Studio conversation transcripts **directly from Dataverse** (the `conversationtranscripts` table, via the Web API with an app registration) **and parses** them into the flat Delta fact tables the **ValueLens — 1905 Extra** PBIP consumes. No manual CSV export step.

Two source modes (set `SOURCE_MODE` in the CONFIG cell):

| `SOURCE_MODE` | Transcript source | Use when |
|---|---|---|
| `'dataverse'` *(default)* | Live Web API pull from `conversationtranscripts`, paged + incremental | Hands-off, scheduled pipeline runs |
| `'files'` | Reads `conversationtranscripts.csv` already in `Files/` | Air-gapped / one-off / no app reg |

The OCV **`feedback`** export remains file-based (it lives in the M365 admin centre, not Dataverse) — drop it in `Files/` or skip it; the notebook tolerates its absence.

```
Dataverse  conversationtranscripts  ──(Web API, app-reg auth, paged)──┐
                                                                       ▼
                          this notebook  (ingest → raw Delta → parse JSON content)
                                                                       ▼
   agent_sessions · agent_turns · agent_errors · agent_subagents
   agent_catalogue · agent_performance      (Delta tables)
                                                                       ▼
                                 PBIP (Lakehouse / Direct Lake)
```

**Output tables** (consumed by the model's *Agent Sessions, Agent Turns, Agent Errors, Agent Sub-Agent Calls, Agent Catalogue,* and *Agent Performance* tables):

| Delta table | Grain |
|---|---|
| `conversationtranscripts_raw` | raw landing — one row per transcript (optional, set `RAW_TABLE=''` to skip) |
| `agent_sessions` | one row per conversation |
| `agent_turns` | one row per user/bot message |
| `agent_errors` | one row per `ErrorTraceData` event |
| `agent_subagents` | one row per `ConnectedAgent` invocation |
| `agent_catalogue` | agent dimension (one row per `botSchemaName`) |
| `agent_performance` | per-conversation KPI fact |

**Permissions (`SOURCE_MODE='dataverse'`)** — an Entra **app registration** added as an **Application User** in the target Dataverse environment, with a security role granting **Read** on the **Conversation Transcript** table (e.g. *System Administrator*, *System Customizer*, or a custom least-privilege role). The notebook authenticates client-credentials against `{DATAVERSE_URL}/.default` — no Microsoft Graph permissions are required for transcripts. (`'files'` mode needs no app reg at all.)

**Pipeline wiring** — this notebook is designed to run as a 4th activity in `CopilotAdoptionPipeline` alongside the three Graph Direct Ingesters. Tag the **CONFIG cell as `parameters`** (cell ⋯ menu → *Toggle parameter cell*) so the pipeline can override `LOOKBACK_DAYS`, `WRITE_MODE`, `SOURCE_MODE`, etc. per run. Schedule it to match your transcript cadence (typically daily, `LOOKBACK_DAYS=1–2` with `WRITE_MODE='append'`; full snapshots use `LOOKBACK_DAYS=0` + `'overwrite'`).


## 1. Configuration  ·  *(mark this as the pipeline `parameters` cell)*

Set `SOURCE_MODE`, then fill the matching block. For `'dataverse'` supply your environment URL + app-registration credentials; for `'files'` only `SOURCE_DIR` matters. `OUTPUT_PREFIX` is the Delta schema (`dbo.` works for both schema-enabled and non-schema Lakehouses).

For a scheduled pipeline run, open this cell's **⋯ menu → Toggle parameter cell** so `CopilotAdoptionPipeline` can override any value (e.g. `LOOKBACK_DAYS`, `WRITE_MODE`, `SOURCE_MODE`) at trigger time. **Production secret handling**: replace the literal `CLIENT_SECRET` with `notebookutils.credentials.getSecret('<kv-uri>', '<secret-name>')`.


In [ ]:
# === CONFIG ===  (tag this cell as the pipeline `parameters` cell)

# --- Source mode ---
SOURCE_MODE   = 'dataverse'   # 'dataverse' = live Web API pull; 'files' = read CSV already in Files/

# --- Dataverse environment(s) — used when SOURCE_MODE='dataverse' ---
# ONE environment:   set DATAVERSE_URL and leave DATAVERSE_URLS = [].
# MANY environments: set DATAVERSE_URLS to a list of env URLs (DATAVERSE_URL is then ignored).
#   The app registration must be added as an Application User in EVERY listed environment.
DATAVERSE_URL  = 'https://orgXXXXXXXX.crm.dynamics.com'   # single environment URL, NO trailing slash
DATAVERSE_URLS = [
    # 'https://org1.crm.dynamics.com',
    # 'https://org2.crm.dynamics.com',
]
TAG_ENVIRONMENT = True   # add a 'SourceEnvironment' column to every row (lets the PBIP slice by env)

TENANT_ID     = '<your-tenant-guid>'
CLIENT_ID     = '<your-app-reg-client-id>'
CLIENT_SECRET = '<your-client-secret-value>'   # prod: notebookutils.credentials.getSecret('<kv-uri>', '<name>')
LOOKBACK_DAYS = 7        # incremental window on conversationstarttime; 0 = full pull (all rows)
PAGE_SIZE     = 5000     # Dataverse OData max page size (server caps at 5000)
RAW_TABLE     = 'dbo.conversationtranscripts_raw'   # raw landing Delta table. The PBIT does NOT
                                                    # need it (agent_* tables are derived below), so on
                                                    # VERY large tenants set RAW_TABLE = '' to skip the
                                                    # content-heavy landing entirely and save time/space.

# Resolve the environment list: DATAVERSE_URLS wins if non-empty, else fall back to the single URL.
# Placeholder/empty entries are dropped so an unset DATAVERSE_URL doesn't get pulled by accident.
DATAVERSE_ENVIRONMENTS = [
    u.rstrip('/') for u in (DATAVERSE_URLS if DATAVERSE_URLS else [DATAVERSE_URL])
    if u and not u.startswith('<') and 'XXXX' not in u
]

# --- Files (fallback transcripts) ---
SOURCE_DIR        = 'Files/copilot_transcripts'
TRANSCRIPTS_FILE  = f'{SOURCE_DIR}/conversationtranscripts.csv'   # used only when SOURCE_MODE='files'

# --- Output ---
OUTPUT_PREFIX = 'dbo'        # Delta schema. Tables: dbo.agent_sessions, dbo.agent_turns, ...
WRITE_MODE    = 'overwrite'  # 'overwrite' = full snapshot; 'append' = add rows; 'merge' = upsert on natural keys (safe multi-env incremental, no duplicates)
TEXT_TRUNCATE = 500          # max chars kept for free-text fields (PII-safe preview)

# --- Large-tenant tuning ---
PANDAS_TO_SPARK_CHUNK_ROWS = 20000   # rows per chunk for the slim agent_* tables (pandas -> Spark Delta)
RAW_TABLE_CHUNK_ROWS       = 2000    # SMALLER chunk for the raw landing: each row carries the full
                                     # conversation `content` JSON (tens of KB), so a 20k-row task can
                                     # exceed spark.rpc.message.maxSize. 2k keeps each task well under it.

# --- Pull resilience & parallelism (large / customer tenant pulls) ---
REQUEST_TIMEOUT    = 120     # per-page HTTP timeout (seconds) for each Dataverse GET
MAX_RETRIES        = 5       # transient-failure retries per request (429 / 5xx / timeout / conn-reset)
RETRY_BACKOFF      = 2.0     # exponential backoff base: sleep ~= RETRY_BACKOFF**attempt (+ jitter); honours Retry-After
MAX_PARALLEL_PULLS = 4       # concurrent fetch workers across (environment x date-window). 1 = fully serial.
PULL_WINDOWS       = 4       # split the LOOKBACK_DAYS range into this many date-windows pulled IN PARALLEL.
                             # Only applies when LOOKBACK_DAYS > 0. A full pull (LOOKBACK_DAYS=0) streams
                             # serially per environment (still retry-hardened, just one window).

## 1b. Preflight  ·  *(run this FIRST in `dataverse` mode — fails fast with a clear verdict)*

A 5-second check before the full pull: gets a client-credentials **token**, then does a single `$top=1` GET on `conversationtranscripts`. It tells you exactly which of the three things is wrong (credentials / Application User permission / empty table) instead of dying mid-pagination. No-op in `files` mode.


In [ ]:
# === 1b. PREFLIGHT (Dataverse auth + permission check) ===
# Run this on its own BEFORE the full ingest. Checks EVERY environment in DATAVERSE_ENVIRONMENTS.
# Skips silently in 'files' mode.
if SOURCE_MODE != 'dataverse':
    print("SOURCE_MODE='files' — preflight skipped (no Dataverse auth needed).")
elif not DATAVERSE_ENVIRONMENTS:
    raise SystemExit("✗ No valid environments. Set DATAVERSE_URL or DATAVERSE_URLS in CONFIG "
                     "(placeholder/<...>/XXXX values are ignored).")
else:
    import requests

    def _preflight_one(base):
        """Returns (ok: bool, message: str) for a single environment URL."""
        base = base.rstrip('/')
        # 1) Token (scope is per-environment)
        try:
            t = requests.post(
                f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token',
                data={'grant_type': 'client_credentials', 'client_id': CLIENT_ID,
                      'client_secret': CLIENT_SECRET, 'scope': f'{base}/.default'},
                timeout=60)
        except Exception as e:
            return False, f'token request failed to send: {e}'
        if t.status_code != 200:
            err = t.json().get('error', '?') if t.headers.get('content-type','').startswith('application/json') else t.text[:200]
            return False, f"token HTTP {t.status_code}: {err}  (check TENANT_ID/CLIENT_ID/CLIENT_SECRET; 'invalid_client' = bad secret)"
        tok = t.json()['access_token']
        # 2) Single-row GET (tests Application User + table read in THIS environment)
        try:
            r = requests.get(
                f'{base}/api/data/v9.2/conversationtranscripts?$select=conversationtranscriptid&$top=1',
                headers={'Authorization': f'Bearer {tok}', 'Accept': 'application/json',
                         'OData-MaxVersion': '4.0', 'OData-Version': '4.0'}, timeout=60)
        except Exception as e:
            return False, f'data GET failed to send: {e}'
        if r.status_code == 403:
            return False, ("403 Forbidden — token valid but app has no access HERE. Add it as an "
                           "Application User in THIS environment (Power Platform Admin Center → "
                           "Settings → Users + permissions → Application users) with READ on conversationtranscripts.")
        if r.status_code == 404:
            return False, "404 — wrong URL (must be https://<org>.crm.dynamics.com, no trailing slash)."
        if r.status_code != 200:
            return False, f'HTTP {r.status_code}: {r.text[:200]}'
        n = len(r.json().get('value', []))
        return True, ('read OK' + ('' if n else ' — but 0 rows on $top=1 (env may have no transcripts yet)'))

    print(f'Preflight across {len(DATAVERSE_ENVIRONMENTS)} environment(s):\n')
    _results = []
    for _env in DATAVERSE_ENVIRONMENTS:
        ok, msg = _preflight_one(_env)
        _results.append(ok)
        print(f"  {'✓' if ok else '✗'} {_env}\n      {msg}")

    _passed = sum(_results)
    print(f'\n{_passed}/{len(DATAVERSE_ENVIRONMENTS)} passed.')
    if _passed == 0:
        raise SystemExit('✗ PREFLIGHT FAILED — no environment is reachable. Fix the messages above before ingesting.')
    elif _passed < len(DATAVERSE_ENVIRONMENTS):
        print('⚠ Some environments failed — cell 2a will SKIP those and ingest only the ones that passed.')
    else:
        print('✓ PREFLIGHT PASSED — safe to run cell 2a (full ingest).')


## 2. Ingest & load the source data

**2a — Ingest from Dataverse** (runs when `SOURCE_MODE='dataverse'`): **loops over every environment** in `DATAVERSE_ENVIRONMENTS`, authenticates the app registration (token scope is per-environment), pages through `conversationtranscripts` with the OData Web API (incremental filter on `conversationstarttime` when `LOOKBACK_DAYS > 0`), tags each row with `SourceEnvironment` (when `TAG_ENVIRONMENT=True`), concatenates all environments into one frame `tx`, and optionally lands the raw rows to `RAW_TABLE`. Environments that error (e.g. a 403 from a missing Application User) are **skipped with a warning** rather than failing the whole run. In `'files'` mode this cell is a no-op and `tx` is read from CSV in 2b.

**2b — Parse JSON content**: locate the `content` column, parse each transcript's JSON, and keep the rows that parse — producing the `parsed` frame the rest of the notebook consumes unchanged. `SourceEnvironment` (if added) rides along through the parse. Transcript volumes are conversation-level (thousands–tens-of-thousands of rows), so a driver-side parse is fine; for very large tenants switch the parse loop to `df.rdd.mapPartitions(...)`.


In [ ]:
# === 2a. INGEST FROM DATAVERSE (Web API) — parallel windows + retry/backoff ===
# Loops over every environment in DATAVERSE_ENVIRONMENTS. For an incremental pull (LOOKBACK_DAYS > 0)
# each environment's date range is split into PULL_WINDOWS half-open slices fetched CONCURRENTLY (up to
# MAX_PARALLEL_PULLS workers across environment x window). Every page GET is wrapped in exponential
# backoff/retry (429 / 5xx / timeout / conn-reset, honouring Retry-After) so a large pull won't crash on
# a single transient timeout. Rows are tagged with SourceEnvironment (when TAG_ENVIRONMENT) and all
# slices are concatenated (de-duped on conversationtranscriptid) into one frame `tx`.
import requests, time, random
import pandas as pd
from datetime import datetime, timezone, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

_SELECT = ('conversationtranscriptid,content,conversationstarttime,schemaversion,'
           'schematype,name,metadata,statecode,statuscode,createdon')
# AGENT IDENTITY: the bot lookup carries the real Copilot Studio agent (schemaname). Expand it so
# single-agent transcripts (no ConnectedAgentInitializeTraceData in content) still resolve a primary
# agent. Falls back to no-expand per slice if the nav property is unavailable in that environment.
_EXPAND = 'bot_conversationtranscriptid($select=schemaname,name,botid)'
_RETRYABLE_STATUS = {429, 500, 502, 503, 504}

def _dataverse_token(base):
    """Client-credentials token for one environment ({base}/.default)."""
    resp = requests.post(
        f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token',
        data={'grant_type': 'client_credentials', 'client_id': CLIENT_ID,
              'client_secret': CLIENT_SECRET, 'scope': f'{base.rstrip("/")}/.default'},
        timeout=60)
    resp.raise_for_status()
    return resp.json()['access_token']

def _get_with_retry(url, headers):
    """GET one page with exponential backoff on transient failures (timeout / conn-reset / 429 / 5xx).
    Honours Retry-After when present. Returns the response for success AND non-retryable HTTP codes
    (e.g. 400/401/403/404) so the caller can inspect them; raises only after MAX_RETRIES."""
    _max     = int(globals().get('MAX_RETRIES', 5))
    _backoff = float(globals().get('RETRY_BACKOFF', 2.0))
    _timeout = int(globals().get('REQUEST_TIMEOUT', 120))
    last = None
    for attempt in range(_max + 1):
        wait = None
        try:
            r = requests.get(url, headers=headers, timeout=_timeout)
            if r.status_code not in _RETRYABLE_STATUS:
                return r
            last = requests.HTTPError(f'HTTP {r.status_code}', response=r)
            ra = r.headers.get('Retry-After')
            if ra:
                try:
                    wait = float(ra)
                except ValueError:
                    wait = None
        except (requests.Timeout, requests.ConnectionError) as e:
            last = e
        if attempt >= _max:
            break
        if wait is None:
            wait = (_backoff ** attempt) + random.uniform(0, 0.75)
        time.sleep(min(wait, 120))
    if last is None:
        raise RuntimeError('request failed after retries')
    raise last

def _fetch_window(base, headers, win_start, win_end):
    """Serial nextLink paging for ONE (environment, [win_start, win_end)) date slice.
    win_start / win_end are ISO-Z strings or None (None = unbounded on that side)."""
    base = base.rstrip('/')
    flt = []
    if win_start:
        flt.append(f'conversationstarttime ge {win_start}')
    if win_end:
        flt.append(f'conversationstarttime lt {win_end}')
    filter_q = ('&$filter=' + ' and '.join(flt)) if flt else ''
    expand = True
    url = (f'{base}/api/data/v9.2/conversationtranscripts?$select={_SELECT}'
           f'&$expand={_EXPAND}{filter_q}')
    rows = []
    while url:
        r = _get_with_retry(url, headers)
        if r.status_code == 400 and expand:
            # bot lookup nav property not exposed in this env -> retry this slice without expand,
            # PRESERVING the date filter (the old code dropped the filter too).
            expand = False
            url = f'{base}/api/data/v9.2/conversationtranscripts?$select={_SELECT}{filter_q}'
            continue
        r.raise_for_status()
        body = r.json()
        rows.extend(body.get('value', []))
        url = body.get('@odata.nextLink')
    return rows

def _iter_windows():
    """Date slices for the pull. Full pull (LOOKBACK_DAYS=0) or PULL_WINDOWS<=1 -> one unbounded slice.
    Otherwise split [now-LOOKBACK_DAYS, now] into PULL_WINDOWS half-open slices (last one open-ended)."""
    n  = max(1, int(globals().get('PULL_WINDOWS', 1)))
    lb = LOOKBACK_DAYS if ('LOOKBACK_DAYS' in globals() and LOOKBACK_DAYS) else 0
    if not lb or lb <= 0 or n == 1:
        return [(None, None)]
    now   = datetime.now(timezone.utc)
    start = now - timedelta(days=int(lb))
    step  = (now - start) / n
    edges = [start + step * i for i in range(n)] + [now]
    fmt   = lambda d: d.strftime('%Y-%m-%dT%H:%M:%SZ')
    wins  = []
    for i in range(n):
        s = fmt(edges[i])
        e = None if i == n - 1 else fmt(edges[i + 1])   # last slice open-ended -> catches rows up to now
        wins.append((s, e))
    return wins

tx = None
if SOURCE_MODE == 'dataverse':
    if not DATAVERSE_ENVIRONMENTS:
        raise SystemExit("✗ No valid environments. Set DATAVERSE_URL or DATAVERSE_URLS in CONFIG.")
    _windows = _iter_windows()
    _workers = max(1, int(globals().get('MAX_PARALLEL_PULLS', 4)))
    print(f'Ingesting conversationtranscripts from {len(DATAVERSE_ENVIRONMENTS)} environment(s) '
          f'x {len(_windows)} window(s)  (LOOKBACK_DAYS={LOOKBACK_DAYS}, 0=full; '
          f'up to {_workers} parallel) ...\n')

    # Pre-fetch one token per environment SERIALLY so auth failures are clear and AAD isn't hammered.
    _tokens, _bad_envs = {}, {}
    for _env in DATAVERSE_ENVIRONMENTS:
        try:
            _tokens[_env.rstrip('/')] = _dataverse_token(_env)
        except Exception as _e:
            _bad_envs[_env] = _e
            print(f'  ⚠ SKIPPED {_env} — token failed: {_e}  (run the preflight to diagnose)')

    def _headers_for(_env):
        return {
            'Authorization':   f'Bearer {_tokens[_env]}',
            'Accept':          'application/json',
            'OData-MaxVersion':'4.0',
            'OData-Version':   '4.0',
            'Prefer':          f'odata.maxpagesize={PAGE_SIZE}',
        }

    _tasks = [(_env, ws, we) for _env in _tokens for (ws, we) in _windows]

    def _run(task):
        _e, ws, we = task
        return _e, (ws, we), _fetch_window(_e, _headers_for(_e), ws, we)

    _env_rows = {_env: [] for _env in _tokens}
    _errs = []
    _n = max(1, min(_workers, len(_tasks) or 1))
    with ThreadPoolExecutor(max_workers=_n) as _ex:
        _futs = {_ex.submit(_run, _t): _t for _t in _tasks}
        for _fut in as_completed(_futs):
            _t = _futs[_fut]
            try:
                _e2, _win, _rows = _fut.result()
                _env_rows[_e2].extend(_rows)
                print(f'    {_t[0]}  [{_t[1] or "start"} .. {_t[2] or "now"}]: +{len(_rows):,}')
            except Exception as _ex2:
                _errs.append((_t, _ex2))
                print(f'    ⚠ window FAILED {_t[0]} [{_t[1]} .. {_t[2]}]: {_ex2}')

    _frames = []
    for _env, _rows in _env_rows.items():
        if not _rows:
            print(f'    {_env}: 0 rows'); continue
        _df = pd.DataFrame(_rows).fillna('')
        _df = _df.rename(columns={col: col.lower() for col in _df.columns})
        # Drop OData annotation columns (e.g. '@odata.etag') that aren't real fields.
        _df = _df[[col for col in _df.columns if not col.startswith('@') and not col.startswith('_')]]
        # Half-open windows don't overlap, but de-dup defensively on the natural key.
        if 'conversationtranscriptid' in _df.columns:
            _df = _df.drop_duplicates(subset=['conversationtranscriptid'])
        if TAG_ENVIRONMENT:
            _df['SourceEnvironment'] = _env
        print(f'    {_env}: {len(_df):,} rows total')
        _frames.append(_df)

    tx = pd.concat(_frames, ignore_index=True) if _frames else pd.DataFrame()
    print(f'\nTotal conversationtranscripts pulled across environments: {len(tx):,}  '
          f'(windows failed: {len(_errs)}, envs skipped: {len(_bad_envs)})')
    if _errs and not len(tx):
        raise SystemExit('✗ All window pulls failed — see the errors above (check network / throttling / preflight).')
    if RAW_TABLE and len(tx):
        # Avoid giant Spark task serialization from one huge pandas->Spark conversion.
        # The raw `content` JSON is heavy, so use the SMALLER raw chunk size: a 20k-row task of
        # full-conversation JSON can exceed spark.rpc.message.maxSize (the customer's 469MB error).
        _chunk_rows = int(globals().get('RAW_TABLE_CHUNK_ROWS', 2000))
        if WRITE_MODE == 'overwrite':
            spark.sql(f'DROP TABLE IF EXISTS {RAW_TABLE}')
        _raw_pdf = tx.astype(str).where(pd.notna(tx), None)
        _raw_schema = StructType([StructField(c, StringType(), True) for c in _raw_pdf.columns])
        _chunk_count = 0
        for _i in range(0, len(_raw_pdf), _chunk_rows):
            _part = _raw_pdf.iloc[_i:_i + _chunk_rows]
            _mode = 'append' if (WRITE_MODE == 'overwrite' or _chunk_count > 0) else WRITE_MODE
            (spark.createDataFrame(_part, schema=_raw_schema)
                .write.mode(_mode)
                .option('overwriteSchema', 'true')
                .option('delta.columnMapping.mode', 'name')
                .option('delta.minReaderVersion', '2')
                .option('delta.minWriterVersion', '5')
                .format('delta').saveAsTable(RAW_TABLE))
            _chunk_count += 1
        print(f'  raw landed -> {RAW_TABLE} ({WRITE_MODE}, chunks={_chunk_count}, chunk_rows={_chunk_rows})')
else:
    print("SOURCE_MODE='files' — transcripts will be read from CSV in cell 2b.")

In [ ]:
# === 2b. PARSE JSON CONTENT ===
import json, re
from datetime import datetime, timezone
import pandas as pd

def _read_csv(path):
    return (spark.read
            .option('header', True)
            .option('multiLine', True)
            .option('escape', '"')
            .option('encoding', 'UTF-8')
            .csv(path))

# `tx` is populated by cell 2a when SOURCE_MODE='dataverse'.
# In 'files' mode (or if 2a was skipped) read the CSV here instead.
if SOURCE_MODE == 'files' or 'tx' not in dir() or tx is None:
    transcripts_sdf = _read_csv(TRANSCRIPTS_FILE)
    transcripts_sdf = transcripts_sdf.toDF(*[c.lower() for c in transcripts_sdf.columns])
    tx = transcripts_sdf.toPandas().fillna('')

print(f'conversationtranscripts rows: {len(tx):,}')

content_col = next((c for c in tx.columns if c.lower().endswith('content')), None)
if content_col is None and len(tx) == 0:
    print('  conversationtranscripts returned 0 rows -> writing empty agent tables (schema preserved).')
elif content_col is None:
    raise RuntimeError('No content column found in conversationtranscripts')

def _safe_json(v):
    if not v:
        return None
    try:
        return json.loads(v)
    except Exception:
        return None

if content_col is None:
    parsed = tx.iloc[0:0].copy()
    parsed['content_json'] = pd.Series([], dtype=object)
else:
    tx['content_json'] = tx[content_col].map(_safe_json)
    parsed = tx[tx['content_json'].notna()].copy()
    # Drop structurally-corrupt rows. A malformed CSV export (embedded commas/quotes in the JSON
    # content) can shift fields so a few rows have a parseable 'content' but garbage in the other
    # columns (e.g. conversationtranscriptid = 'role":0}'). A real id is a GUID; rows whose id
    # isn't a GUID are export corruption and would otherwise fail datetime/number typing in the PBIP.
    _idcol = next((c for c in parsed.columns if c.lower() == 'conversationtranscriptid'), None)
    if _idcol is not None and len(parsed):
        _guid = re.compile(r'^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$')
        _before = len(parsed)
        parsed = parsed[parsed[_idcol].map(lambda v: bool(_guid.match(str(v).strip())))].copy()
        if _before != len(parsed):
            print(f'  dropped {_before - len(parsed):,} structurally-corrupt row(s) (non-GUID transcript id)')
print(f'rows with valid JSON content: {len(parsed):,}')

# --- AGENT IDENTITY from the bot lookup (reliable even when content JSON is corrupt) ---
# The Dataverse $expand yields a nested 'bot_conversationtranscriptid' dict; the CSV export
# carries a flat 'bot_conversationtranscriptid.schemaname' column. Surface one cleaned column.
def _clean_schema(val):
    """Keep a bot schemaname only if it looks like a real schema token, not CSV-corruption
    bleed (some malformed CSV exports leak JSON/content into this column). Reject JSON chars + whitespace."""
    if val is None:
        return None
    s = str(val).strip()
    if not s or len(s) > 128:
        return None
    if any(ch in s for ch in ('{', '}', '"', ':', '\n', '\t', ' ', ',', '<', '>', '[', ']')):
        return None
    return s

def _row_schema(row):
    for c in ('bot_conversationtranscriptid.schemaname', 'bot_schemaname', 'schemaname'):
        if c in row and pd.notna(row.get(c)):
            v = _clean_schema(row.get(c))
            if v:
                return v
    nb = row.get('bot_conversationtranscriptid')
    if isinstance(nb, dict):
        return _clean_schema(nb.get('schemaname'))
    return None

if len(parsed):
    parsed['agent_schema_hint'] = parsed.apply(_row_schema, axis=1)
else:
    parsed['agent_schema_hint'] = pd.Series([], dtype=object)
print(f'rows with a bot schemaname hint: {int(parsed["agent_schema_hint"].notna().sum()) if len(parsed) else 0:,}')



## 3. Parsing helpers

Reused verbatim from `parse_copilot_transcripts.py`. **PII handling**: raw AAD `objectId` is emitted (needed to join to `Chat + Agent Org Data`[id] for Organization slicing inside the governed Lakehouse); free-text profile fields are dropped; message bodies truncated to `TEXT_TRUNCATE` chars.

In [ ]:
def keep_user_id(v):
    """Return the raw AAD objectId unchanged so users stay joinable downstream.

    NOTE: despite the legacy output column name ``user_id_hash``, this value is NOT
    hashed - it is the raw user id, kept intentionally for cross-source joins.
    Empty -> None so joins stay clean.
    """
    if v is None or v == '':
        return None
    return str(v)

# Backwards-compatible alias (existing cells call hash_id); same pass-through, no hashing.
hash_id = keep_user_id

def get_activities(content):
    if isinstance(content, dict):
        for k in ('activities', 'Activities', 'transcript', 'messages'):
            if k in content and isinstance(content[k], list):
                return content[k]
    if isinstance(content, list):
        return content
    return []

def extract_agent_name(act_list):
    """Returns (primary_agent_schema, [connected_agent_schemas...])."""
    primary = None
    connected = []
    for a in act_list:
        if not isinstance(a, dict):
            continue
        v = a.get('value') or {}
        if not isinstance(v, dict):
            continue
        if a.get('valueType') == 'ConnectedAgentInitializeTraceData':
            sn = v.get('connectedAgentBotSchemaName') or v.get('botSchemaName')
            if sn and sn not in connected:
                connected.append(sn)
            if primary is None:
                primary = v.get('parentBotSchemaName')
        # Single-agent transcripts have no ConnectedAgent trace; pick up a botSchemaName anywhere.
        elif primary is None:
            bsn = v.get('botSchemaName') or v.get('parentBotSchemaName')
            if bsn:
                primary = bsn
    return primary, connected

def _verdict_from_value(val):
    if val is None:
        return None
    if isinstance(val, bool):
        return 'Positive' if val else 'Negative'
    if isinstance(val, (int, float)):
        return 'Positive' if val > 0 else 'Negative'
    s = str(val).strip().lower()
    if s in ('true','1','like','thumbsup','thumbs up','up','positive','helpful','yes','good','smile'):
        return 'Positive'
    if s in ('false','0','dislike','thumbsdown','thumbs down','down','negative','unhelpful','no','bad','frown'):
        return 'Negative'
    return None

def extract_submitted_feedback(act_list):
    """Scan every canonical schema position where a completed feedback verdict
    writes back into a transcript activity stream. Returns (verdict, comment)."""
    verdict = None
    comment = None
    for a in act_list:
        if not isinstance(a, dict):
            continue
        for rx in (a.get('reactionsAdded') or []):
            if isinstance(rx, dict):
                v = _verdict_from_value(rx.get('type') or rx.get('reactionType'))
                if v:
                    verdict = v
        cd = a.get('channelData') if isinstance(a.get('channelData'), dict) else {}
        fl = cd.get('feedbackLoop') if isinstance(cd.get('feedbackLoop'), dict) else {}
        if fl:
            v = _verdict_from_value(fl.get('value') or fl.get('reaction') or fl.get('vote'))
            if v:
                verdict = v
            comment = comment or fl.get('feedbackText') or fl.get('text')
        cf = cd.get('feedback') if isinstance(cd.get('feedback'), dict) else {}
        if cf:
            v = _verdict_from_value(cf.get('helpful') if 'helpful' in cf else cf.get('reaction'))
            if v:
                verdict = v
            comment = comment or cf.get('feedbackText') or cf.get('text')
        val = a.get('value') if isinstance(a.get('value'), dict) else {}
        if val:
            action = str(val.get('actionName') or a.get('name') or '').lower()
            fb = val.get('feedback') if isinstance(val.get('feedback'), dict) else None
            if fb or 'feedback' in action or 'submitaction' in action:
                src = fb or val
                v = _verdict_from_value(src.get('helpful') if 'helpful' in src else (src.get('reaction') or src.get('vote') or src.get('rating')))
                if v:
                    verdict = v
                comment = comment or src.get('feedbackText') or src.get('text')
    return verdict, comment

def _iso(ms):
    return datetime.fromtimestamp(ms/1000, tz=timezone.utc).isoformat() if ms else None

def _clean_token(val, maxlen=200):
    """Clean a candidate agent label/id. Unlike _clean_schema this ALLOWS spaces
    (display names are multi-word) but still rejects CSV-corruption bleed."""
    if val is None:
        return None
    s = str(val).strip()
    if not s or len(s) > maxlen:
        return None
    if any(ch in s for ch in ('{', '}', '"', '\n', '\t', '<', '>', '[', ']')):
        return None
    return s

def resolve_agent_identity(acts, row):
    """Resolve (agent_schema, agent_name, agent_id, connected_schemas) for a transcript
    using the most reliable signal available, in priority order:
      1. Dataverse bot lookup  (schemaname / name / botid)        - survives corrupt content
      2. Bot-message from.name / from.id                          - present in almost every convo
      3. User-message recipient.name / recipient.id               - the agent as addressed
      4. Content orchestration trace botSchemaName / parentBot... - multi-agent traces only
      5. Transcript-level `name` field                            - last-resort label

    Why this exists: the previous logic resolved identity ONLY from the bot-lookup
    schemaname plus connected-agent orchestration traces. Single-agent transcripts have
    neither, so primary_agent_schema was null for the bulk of conversations -> sessions
    unattributed and the self-sourced agent_catalogue collapsed to a handful of agents.
    The bot's from/recipient identity is present in nearly every conversation, so we use it.
    """
    # --- connected (sub-)agents + any parent/bot schema from orchestration traces ---
    connected, trace_schema = [], None
    for a in acts:
        if not isinstance(a, dict):
            continue
        v = a.get('value') or {}
        if not isinstance(v, dict):
            continue
        if a.get('valueType') == 'ConnectedAgentInitializeTraceData':
            sn = v.get('connectedAgentBotSchemaName') or v.get('botSchemaName')
            if sn and sn not in connected:
                connected.append(sn)
            if trace_schema is None:
                trace_schema = v.get('parentBotSchemaName')
        elif trace_schema is None:
            bsn = v.get('botSchemaName') or v.get('parentBotSchemaName')
            if bsn:
                trace_schema = bsn

    # --- bot lookup (nested dict from $expand, or flat columns from a CSV export) ---
    lookup_schema = row.get('agent_schema_hint')  # pre-cleaned schema token (cell 2b)
    lookup_name = lookup_id = None
    nbref = row.get('bot_conversationtranscriptid')
    if isinstance(nbref, dict):
        lookup_schema = lookup_schema or _clean_schema(nbref.get('schemaname'))
        lookup_name = _clean_token(nbref.get('name'))
        lookup_id = _clean_token(nbref.get('botid'))
    for c in ('bot_conversationtranscriptid.name', 'bot_name'):
        if not lookup_name and c in row:
            lookup_name = _clean_token(row.get(c))
    for c in ('bot_conversationtranscriptid.botid', 'botid', 'bot_botid'):
        if not lookup_id and c in row:
            lookup_id = _clean_token(row.get(c))

    # --- from/recipient identity on the message stream ---
    msg_name = msg_id = None
    for a in acts:
        if not isinstance(a, dict) or a.get('type') != 'message':
            continue
        frm = a.get('from') or {}
        if frm.get('role') in (1, 'bot'):
            msg_name = msg_name or _clean_token(frm.get('name'))
            msg_id = msg_id or _clean_token(frm.get('id'))
        else:
            rcp = a.get('recipient') or {}
            if isinstance(rcp, dict):
                msg_name = msg_name or _clean_token(rcp.get('name'))
                msg_id = msg_id or _clean_token(rcp.get('id'))
        if msg_name and msg_id:
            break

    tx_name = _clean_token(row.get('name'))

    schema = lookup_schema or trace_schema
    name = lookup_name or msg_name or tx_name
    bot_id = lookup_id or msg_id
    # Keep the relationship key (agent_schema) populated even when no schema token exists,
    # so the session is attributed and the agent appears in the catalogue. Prefer a human
    # name over a raw id for the fallback key (the catalogue humanises it for display).
    if not schema:
        schema = name or bot_id

    return schema, name, bot_id, connected

print('helpers loaded')

## 4. Build per-session fact table — `agent_sessions`

In [ ]:
def build_sessions(parsed):
    rows = []
    for _, r in parsed.iterrows():
        acts = get_activities(r['content_json'])
        msgs = [a for a in acts if isinstance(a, dict) and a.get('type') == 'message']
        user_msgs = [a for a in msgs if (a.get('from') or {}).get('role') in (0, 'user')]
        bot_msgs  = [a for a in msgs if (a.get('from') or {}).get('role') in (1, 'bot')]
        user_id_raw = None
        for a in msgs:
            if (a.get('from') or {}).get('role') in (0, 'user'):
                user_id_raw = (a.get('from') or {}).get('aadObjectId') or (a.get('from') or {}).get('id')
                break
        primary_agent, agent_name, agent_bot_id, connected_agents = resolve_agent_identity(acts, r)
        cost = latency = plan_steps = 0
        for a in acts:
            v = a.get('value') if isinstance(a, dict) else None
            if isinstance(v, dict) and 'displayedCost' in v:
                try:
                    cost += int(v.get('displayedCost') or 0)
                except (TypeError, ValueError):
                    pass
                plan_steps += 1
                et = v.get('executionTime')
                if et:
                    m = re.match(r'(\d+):(\d+):(\d+\.?\d*)', str(et))
                    if m:
                        h, mn, s = int(m.group(1)), int(m.group(2)), float(m.group(3))
                        latency += int((h*3600 + mn*60 + s) * 1000)
        k_searched = k_answered = False
        k_sources = 0
        for a in acts:
            if isinstance(a, dict) and a.get('valueType') == 'KnowledgeTraceData':
                v = a.get('value') or {}
                if isinstance(v, dict):
                    if v.get('isKnowledgeSearched'):
                        k_searched = True
                    if v.get('completionState') == 'Answered':
                        k_answered = True
                    k_sources += len(v.get('citedKnowledgeSources') or [])
        # --- grounding source: did the agent answer from internal knowledge, the web, or neither ---
        g_internal = g_external = False
        for a in acts:
            if not isinstance(a, dict):
                continue
            vt = a.get('valueType'); nm = str(a.get('name') or ''); gv = a.get('value') or {}
            if vt == 'KnowledgeTraceData' and isinstance(gv, dict) and gv.get('isKnowledgeSearched'):
                g_internal = True
            if 'UniversalSearch' in nm and isinstance(gv, dict):
                if gv.get('knowledgeSources'):
                    g_internal = True
                for ep in ((gv.get('searchContextTraceInfo') or {}).get('endpoints') or []):
                    h = str(ep).lower()
                    if 'bing.com' in h or 'bing.net' in h:
                        g_external = True
                    elif 'sharepoint' in h or 'onedrive' in h or 'crm.dynamics' in h or 'graph.microsoft' in h:
                        g_internal = True
                    elif h.startswith('http'):
                        g_external = True
            if 'web' in nm.lower() and 'search' in nm.lower():
                g_external = True
        grounding_source = ('Mixed (Internal+Web)' if g_internal and g_external
                            else 'External (Web)' if g_external
                            else 'Internal' if g_internal
                            else 'Ungrounded')
        errors = [ (a.get('value') or {}).get('errorCode') or 'Unknown'
                   for a in acts if isinstance(a, dict) and a.get('valueType') == 'ErrorTraceData' ]
        feedback_offered = sum(1 for a in acts if isinstance(a, dict)
                               and isinstance(a.get('channelData'), dict)
                               and 'feedbackLoop' in a.get('channelData', {}))
        verdict, comment = extract_submitted_feedback(acts)
        first_ts = last_ts = None
        for a in acts:
            ts = a.get('timestampMs') if isinstance(a, dict) else None
            if ts:
                first_ts = ts if first_ts is None or ts < first_ts else first_ts
                last_ts  = ts if last_ts  is None or ts > last_ts  else last_ts
        duration_ms = (last_ts - first_ts) if first_ts and last_ts else None
        session_start = _iso(first_ts) or r.get('conversationstarttime', '')
        first_prompt = ''
        for a in user_msgs:
            if a.get('text', ''):
                first_prompt = a['text'][:TEXT_TRUNCATE]
                break
        rows.append({
            'conversation_id': r.get('conversationtranscriptid', ''),
            'session_start_utc': session_start,
            'session_duration_ms': duration_ms,
            'user_id_hash': hash_id(user_id_raw),
            'primary_agent_schema': primary_agent,
            'connected_agent_schemas': '|'.join(connected_agents) if connected_agents else None,
            'connected_agent_count': len(connected_agents),
            'agent_name': agent_name,
            'agent_id': agent_bot_id,
            'msg_count': len(msgs),
            'user_msg_count': len(user_msgs),
            'bot_msg_count': len(bot_msgs),
            'plan_step_count': plan_steps,
            'total_displayed_cost': cost,
            'total_latency_ms': latency,
            'knowledge_searched': k_searched,
            'knowledge_answered': k_answered,
            'knowledge_sources_count': k_sources,
            'grounding_source': grounding_source,
            'error_count': len(errors),
            'first_error_code': errors[0] if errors else None,
            'feedback_offered_count': feedback_offered,
            'feedback_submitted': 1 if verdict else 0,
            'feedback_verdict': verdict,
            'feedback_comment': comment,
            'first_user_prompt': first_prompt,
            # Outcome / topic columns: explicit outcome is set by Copilot Studio only when
            # configured; left null here (the dashboard's DAX derives outcome/topics). These
            # columns must exist so the model's measures bind. Matches the M parser contract.
            'session_outcome_explicit': None,
            'session_outcome_reason_explicit': None,
            'is_authenticated': bool(user_id_raw),
            'is_returning_user': None,        # filled in below (user appears in >1 conversation)
            'primary_topic_derived': None,
        })
    _df = pd.DataFrame(rows, columns=['conversation_id', 'session_start_utc', 'session_duration_ms', 'user_id_hash', 'primary_agent_schema', 'connected_agent_schemas', 'connected_agent_count', 'agent_name', 'agent_id', 'msg_count', 'user_msg_count', 'bot_msg_count', 'plan_step_count', 'total_displayed_cost', 'total_latency_ms', 'knowledge_searched', 'knowledge_answered', 'knowledge_sources_count', 'grounding_source', 'error_count', 'first_error_code', 'feedback_offered_count', 'feedback_submitted', 'feedback_verdict', 'feedback_comment', 'first_user_prompt', 'session_outcome_explicit', 'session_outcome_reason_explicit', 'is_authenticated', 'is_returning_user', 'primary_topic_derived'])
    # is_returning_user: a user_id_hash that appears in more than one conversation.
    _vc = _df['user_id_hash'].value_counts()
    _returning = set(_vc[_vc > 1].index) - {None, ''}
    _df['is_returning_user'] = _df['user_id_hash'].apply(
        lambda u: True if (u is not None and u != '' and u in _returning) else False)
    return _df

sessions = build_sessions(parsed)
print(f'sessions: {len(sessions):,}')

## 5. Build per-turn fact table — `agent_turns`

In [ ]:
def build_turns(parsed):
    rows = []
    for _, r in parsed.iterrows():
        conv_id = r.get('conversationtranscriptid', '')
        acts = get_activities(r['content_json'])
        intent_by_reply, knowledge_by_reply = {}, {}
        for a in acts:
            if isinstance(a, dict) and a.get('valueType') == 'IntentRecognition':
                v = a.get('value') or {}
                rid = a.get('replyToId')
                if rid and isinstance(v, dict):
                    intent_by_reply[rid] = {
                        'intent_title': v.get('intentTitle'),
                        'intent_id': v.get('intentId'),
                        'intent_score': (v.get('intentScore') or {}).get('score'),
                    }
            if isinstance(a, dict) and a.get('valueType') == 'KnowledgeTraceData':
                v = a.get('value') or {}
                rid = a.get('replyToId')
                if rid and isinstance(v, dict):
                    knowledge_by_reply[rid] = {
                        'knowledge_searched': bool(v.get('isKnowledgeSearched')),
                        'knowledge_answered': v.get('completionState') == 'Answered',
                        'knowledge_sources': '|'.join(v.get('citedKnowledgeSources') or []),
                    }
        for a in acts:
            if not isinstance(a, dict) or a.get('type') != 'message':
                continue
            role = (a.get('from') or {}).get('role')
            role_label = 'user' if role in (0, 'user') else ('bot' if role in (1, 'bot') else 'other')
            ts_ms = a.get('timestampMs')
            from_id = (a.get('from') or {}).get('aadObjectId') or (a.get('from') or {}).get('id')
            msg_id = a.get('id')
            intent = intent_by_reply.get(msg_id, {})
            knowledge = knowledge_by_reply.get(msg_id, {})
            cd = a.get('channelData') or {}
            fb_offered = isinstance(cd, dict) and 'feedbackLoop' in cd
            turn_verdict, turn_comment = extract_submitted_feedback([a])
            rows.append({
                'conversation_id': conv_id,
                'turn_id': msg_id,
                'turn_timestamp_utc': _iso(ts_ms),
                'turn_role': role_label,
                'turn_channel': a.get('channelId'),
                'from_user_hash': hash_id(from_id) if role_label == 'user' else None,
                'text_preview_500': (a.get('text') or '')[:TEXT_TRUNCATE],
                'text_length': len(a.get('text') or ''),
                'intent_title': intent.get('intent_title'),
                'intent_id': intent.get('intent_id'),
                'intent_score': intent.get('intent_score'),
                'knowledge_searched': knowledge.get('knowledge_searched'),
                'knowledge_answered': knowledge.get('knowledge_answered'),
                'knowledge_sources': knowledge.get('knowledge_sources'),
                'feedback_offered': fb_offered,
                'feedback_verdict': turn_verdict,
                'feedback_comment': turn_comment,
            })
    return pd.DataFrame(rows, columns=['conversation_id', 'turn_id', 'turn_timestamp_utc', 'turn_role', 'turn_channel', 'from_user_hash', 'text_preview_500', 'text_length', 'intent_title', 'intent_id', 'intent_score', 'knowledge_searched', 'knowledge_answered', 'knowledge_sources', 'feedback_offered', 'feedback_verdict', 'feedback_comment'])

turns = build_turns(parsed)
print(f'turns: {len(turns):,}')

## 6. Errors + sub-agent invocations — `agent_errors`, `agent_subagents`

In [ ]:
def build_errors(parsed):
    rows = []
    for _, r in parsed.iterrows():
        for a in get_activities(r['content_json']):
            if not isinstance(a, dict) or a.get('valueType') != 'ErrorTraceData':
                continue
            v = a.get('value') or {}
            rows.append({
                'conversation_id': r.get('conversationtranscriptid', ''),
                'error_timestamp_utc': _iso(a.get('timestampMs')),
                'error_code': v.get('errorCode'),
                'error_sub_code': v.get('errorSubCode'),
                'error_message': (v.get('errorMessage') or '')[:TEXT_TRUNCATE],
                'is_user_error': bool(v.get('isUserError')),
            })
    return pd.DataFrame(rows, columns=['conversation_id', 'error_timestamp_utc', 'error_code', 'error_sub_code', 'error_message', 'is_user_error'])

def build_subagents(parsed):
    rows = []
    for _, r in parsed.iterrows():
        for a in get_activities(r['content_json']):
            if not isinstance(a, dict):
                continue
            vt = a.get('valueType')
            if vt not in ('ConnectedAgentInitializeTraceData', 'ConnectedAgentCompletedTraceData'):
                continue
            v = a.get('value') or {}
            rows.append({
                'conversation_id': r.get('conversationtranscriptid', ''),
                'invocation_timestamp_utc': _iso(a.get('timestampMs')),
                'event_type': 'initialize' if vt.endswith('InitializeTraceData') else 'completed',
                'parent_agent_schema': v.get('parentBotSchemaName'),
                'connected_agent_schema': v.get('connectedAgentBotSchemaName') or v.get('botSchemaName'),
                'dialog_schema': v.get('dialogSchemaName'),
                'user_id_hash': hash_id(v.get('userId')),
                'plan_step_id': v.get('planStepId'),
            })
    return pd.DataFrame(rows, columns=['conversation_id', 'invocation_timestamp_utc', 'event_type', 'parent_agent_schema', 'connected_agent_schema', 'dialog_schema', 'user_id_hash', 'plan_step_id'])

errors = build_errors(parsed)
subagents = build_subagents(parsed)
print(f'errors: {len(errors):,}  |  sub-agent events: {len(subagents):,}')

## 7. Agent catalogue (dimension) — `agent_catalogue`

In [ ]:
import re

# Curated friendly labels. Extend per customer: 'schemaname': 'Friendly Name'.
AGENT_DESCRIPTIONS = {
    'msdyn_copilotforemployeeselfservicehr': 'Employee Self-Service HR',
    'msdyn_copilotforemployeeselfserviceit': 'Employee Self-Service IT',
    'msdyn_copilotforemployeeselfservicefinance': 'Employee Self-Service Finance',
    'msdyn_copilotforemployeeselfservicegeneral': 'Employee Self-Service General',
}

# Orchestrator / header agents all collapse to ONE label so they cluster together in the
# Agent Evaluation visuals instead of appearing as many "Copilots Header xxxxx" rows.
ORCHESTRATOR_LABEL = 'M365 Copilot Orchestrator'

_PUBLISHER_PREFIX = re.compile(r'^(?:[a-z][a-z0-9]{1,7}|copilots_header|copilots|copilot)_', re.I)
_ACRONYMS = {'BT', 'HR', 'IT', 'AI', 'HQ', 'QA', 'CRM', 'ERP', 'M365', 'API', 'KPI', 'PDF', 'SLA', 'VPN'}

def _is_orchestrator(schema):
    s = str(schema or '').lower()
    return 'header' in s or 'orchestrator' in s

def _selfservice_label(key):
    m = re.search(r'copilotforemployeeselfservice([a-z]+)', key)
    suffix = m.group(1) if m else ''
    if not suffix:
        return 'Employee Self-Service'
    label = suffix.upper() if suffix.upper() in _ACRONYMS else suffix.capitalize()
    return f'Employee Self-Service {label}'

def humanise_schema(schema):
    """Readable label from a raw Dataverse schemaname when no friendly bot name is available."""
    if not schema:
        return 'Unknown agent'
    s = str(schema)
    core = _PUBLISHER_PREFIX.sub('', s)                        # strip publisher prefix (crb93_, cr23c_, msdyn_, ...)
    core = re.sub(r'[_\-]+', ' ', core)
    core = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', core).strip()   # split camelCase where present
    if not core:
        return 'Unknown agent'
    compact = core.replace(' ', '')
    # opaque id (hex blob, or agent+token) -> a clear label rather than a bare token
    if re.fullmatch(r'(?:agent)?[0-9a-f]{4,}', compact, re.I) or re.fullmatch(r'agent[0-9a-z]{3,}', compact, re.I):
        tail = re.sub(r'^agent', '', compact, flags=re.I)
        return f'Agent {tail}' if tail else 'Agent'
    words = []
    for w in core.split():
        if w.upper() in _ACRONYMS:
            words.append(w.upper())
        elif w.isupper():
            words.append(w)
        else:
            words.append(w.capitalize())
    return ' '.join(words) or f'Agent {s}'

def build_agent_dim(sessions, subagents):
    s1 = pd.Series(sessions['primary_agent_schema'].dropna().unique())
    s2 = pd.Series(subagents['connected_agent_schema'].dropna().unique()) if not subagents.empty else pd.Series(dtype=str)
    s3 = pd.Series(subagents['parent_agent_schema'].dropna().unique()) if not subagents.empty else pd.Series(dtype=str)
    all_schemas = pd.concat([s1, s2, s3]).drop_duplicates()
    dim = pd.DataFrame({'agent_schema': sorted(all_schemas.tolist())})
    # Real resolved bot names (from resolve_agent_identity -> sessions['agent_name']).
    _name_map = {}
    if 'agent_name' in sessions.columns:
        for _sc, _nm in zip(sessions['primary_agent_schema'], sessions['agent_name']):
            if _sc is not None and _sc not in _name_map and isinstance(_nm, str) and _nm.strip():
                _name_map[_sc] = _nm.strip()
    def _display(s):
        # orchestrator cluster -> curated -> real bot name -> self-service pattern -> humanised token
        if _is_orchestrator(s):
            return ORCHESTRATOR_LABEL
        key = str(s).lower()
        if key in AGENT_DESCRIPTIONS:
            return AGENT_DESCRIPTIONS[key]
        nm = _name_map.get(s)
        if nm and nm.strip().lower() != key:
            return nm.strip()
        if 'copilotforemployeeselfservice' in key:
            return _selfservice_label(key)
        return humanise_schema(s)
    dim['agent_display_name'] = dim['agent_schema'].map(_display)
    def classify(s):
        s = str(s).lower()
        if 'header' in s or 'orchestrator' in s:
            return 'Orchestrator'
        if 'copilotforemployeeselfservice' in s:
            return 'Self-Service'
        if s.startswith('crb') or s.startswith('crd'):
            return 'Custom Declarative'
        if s.startswith('msdyn'):
            return 'First-Party Connected'
        return 'Other'
    dim['agent_class'] = dim['agent_schema'].map(classify)
    return dim

agent_dim = build_agent_dim(sessions, subagents)
print(f'unique agents: {len(agent_dim):,}')

## 8b. Agent Performance KPI fact — `agent_performance`

Per-conversation KPI table that powers the **Agent Performance / Agent Evaluation** pages. Every column is derived directly from the raw transcript activities, so it **replaces the old `conversation_kpis_flattened.csv`** — no separate analytics extract is needed. `SessionType`, `SessionOutcome`, `OutcomeReason` and `ImpliedSuccess` are documented heuristics; the model's calculated columns (`PromptQualityScore`, `ConversationType`, `ConversationComplexity`, `PromptLength`, `BotFriendlyName`) are computed downstream in DAX and intentionally not produced here.

In [ ]:
def build_agent_performance(parsed):
    rows = []
    for _, r in parsed.iterrows():
        acts = get_activities(r['content_json'])
        msgs = [a for a in acts if isinstance(a, dict) and a.get('type') == 'message']
        user_msgs = [a for a in msgs if (a.get('from') or {}).get('role') in (0, 'user')]
        bot_msgs  = [a for a in msgs if (a.get('from') or {}).get('role') in (1, 'bot')]
        # Robust agent identity (same resolver as build_sessions): keeps BotName keyed
        # consistently with sessions.primary_agent_schema and populates a real BotId.
        primary_agent, _agent_name, _agent_bot_id, connected_agents = resolve_agent_identity(acts, r)

        topics, last_intent_id = [], None
        for a in acts:
            if isinstance(a, dict) and a.get('valueType') == 'IntentRecognition':
                v = a.get('value') or {}
                t = v.get('intentTitle')
                if t and t not in topics:
                    topics.append(t)
                last_intent_id = v.get('intentId') or last_intent_id

        sys_error = user_error = False
        for a in acts:
            if isinstance(a, dict) and a.get('valueType') == 'ErrorTraceData':
                v = a.get('value') or {}
                if v.get('isUserError'):
                    user_error = True
                else:
                    sys_error = True

        latency_total = plan_steps = trace_count = event_count = 0
        for a in acts:
            if not isinstance(a, dict):
                continue
            if a.get('type') == 'trace':
                trace_count += 1
            elif a.get('type') == 'event':
                event_count += 1
            v = a.get('value') if isinstance(a.get('value'), dict) else None
            if v and 'executionTime' in v:
                plan_steps += 1
                m = re.match(r'(\d+):(\d+):(\d+\.?\d*)', str(v.get('executionTime')))
                if m:
                    h, mn, s = int(m.group(1)), int(m.group(2)), float(m.group(3))
                    latency_total += int((h*3600 + mn*60 + s) * 1000)
        avg_latency = int(latency_total / plan_steps) if plan_steps else 0

        first_ts = last_ts = None
        for a in acts:
            ts = a.get('timestampMs') if isinstance(a, dict) else None
            if ts:
                first_ts = ts if first_ts is None or ts < first_ts else first_ts
                last_ts  = ts if last_ts  is None or ts > last_ts  else last_ts
        duration_s = int((last_ts - first_ts) / 1000) if first_ts and last_ts else 0

        def _txt(a):
            return (a.get('text') or '')[:TEXT_TRUNCATE] if isinstance(a, dict) else ''
        first_user = _txt(user_msgs[0]) if user_msgs else ''
        last_user  = _txt(user_msgs[-1]) if user_msgs else ''
        first_bot  = _txt(bot_msgs[0]) if bot_msgs else ''
        last_bot   = _txt(bot_msgs[-1]) if bot_msgs else ''

        upn = None
        for a in acts:
            if not isinstance(a, dict):
                continue
            v = a.get('value') if isinstance(a.get('value'), dict) else {}
            vars_ = v.get('variables') if isinstance(v.get('variables'), dict) else {}
            upn = upn or vars_.get('ESS_UserContext_UPN') or vars_.get('Var_ESS_UserContext_UPN')

        # --- Heuristics ---
        engaged = bool(topics) or len(bot_msgs) > 0
        session_type = 'Engaged' if engaged else 'Unengaged'
        if connected_agents:
            outcome = 'Escalated'
        elif not engaged or (len(user_msgs) > 0 and len(bot_msgs) == 0) or (sys_error and len(bot_msgs) == 0):
            outcome = 'Abandoned'
        else:
            outcome = 'Resolved'
        outcome_reason = ('SystemError' if sys_error else 'UserError' if user_error else
                          ('Escalated' if outcome == 'Escalated' else
                           'Abandoned' if outcome == 'Abandoned' else 'Completed'))
        implied_success = 'TRUE' if (outcome == 'Resolved' and not sys_error) else 'FALSE'

        start_iso = _iso(first_ts) or r.get('conversationstarttime', '')
        end_iso = _iso(last_ts) or ''

        # --- contract columns to match the agent_performance Delta schema (M parser parity) ---
        _perf_errcodes = [ (a.get('value') or {}).get('errorCode')
                           for a in acts if isinstance(a, dict) and a.get('valueType') == 'ErrorTraceData' ]
        _perf_first_err = next((e for e in _perf_errcodes if e), None)
        _perf_authed = any(((a.get('from') or {}).get('aadObjectId')) for a in user_msgs)
        _perf_connected = set()
        for _pa in acts:
            if isinstance(_pa, dict):
                _pv = _pa.get('value') or {}
                if isinstance(_pv, dict) and _pv.get('connectedAgentBotSchemaName'):
                    _perf_connected.add(_pv.get('connectedAgentBotSchemaName'))
        _perf_gen = sum(1 for a in acts if isinstance(a, dict)
                        and a.get('valueType') in ('GenerativeAIResponseTraceData', 'GenerativeResponseTraceData'))
        rows.append({
            'ConversationTranscriptId': r.get('conversationtranscriptid', ''),
            'ConversationStartTime': r.get('conversationstarttime', '') or start_iso,
            'SchemaVersion': r.get('schemaversion', ''),
            'BotName': primary_agent or '',
            'BotId': _agent_bot_id or r.get('botid', ''),
            'AADTenantId': r.get('aadtenantid', ''),
            'BatchId': r.get('batchid', ''),
            'SessionStartTimeUtc': start_iso,
            'SessionEndTimeUtc': end_iso,
            'SessionType': session_type,
            'SessionOutcome': outcome,
            'TurnCount': len(msgs),
            'OutcomeReason': outcome_reason,
            'ImpliedSuccess': implied_success,
            'LastUserIntentId': last_intent_id,
            'IsDesignMode': 'FALSE',
            'Locale': r.get('locale', ''),
            'UserMessageCount': len(user_msgs),
            'BotMessageCount': len(bot_msgs),
            'TotalMessageCount': len(msgs),
            'FirstUserMessage': first_user,
            'LastUserMessage': last_user,
            'FirstBotMessage': first_bot,
            'LastBotMessage': last_bot,
            'TopicsTriggered': '|'.join(topics) if topics else '',
            'TopicCount': len(topics),
            'PrimaryTopic': topics[0] if topics else '',
            'DurationSeconds': duration_s,
            'AverageLatencyMs': avg_latency,
            'Var_ESS_UserContext_UPN': upn,
            'Var_ESS_Message_Disclaimer': '',
            'Var_ESS_UserContext_RefreshInterval_Hours': '',
            'Var_ESS_UserContext_Conversation_Initialized': '',
            'AllVariables': '',
            'TotalActivityCount': len(acts),
            'TraceActivityCount': trace_count,
            'EventActivityCount': event_count,
            'StatusCode': 1 if outcome == 'Resolved' else 0,
            'StateCode': 1,
            'FirstErrorCode': _perf_first_err,
            'ErrorCategory': None,
            'IsAuthenticated': 'TRUE' if _perf_authed else 'FALSE',
            'MultiAgentSession': 'TRUE' if _perf_connected else 'FALSE',
            'SessionOutcomeExplicit': None,
            'SessionOutcomeReasonExplicit': None,
            'GenerativeResponseCount': _perf_gen,
            'LLMCallCount': None,
            'PromptTokenCount': None,
            'CompletionTokenCount': None,
            'TotalTokenCount': None,
            'PluginCallCount': None,
            'PluginSuccessCount': None,
            'TopicsStarted': len(topics),
            'TopicsCompleted': None,
            'TopicCompletionRate': None,
            'PrimaryTopicDwellMs': None,
            'TotalTopicDwellMs': None,
        })
    return pd.DataFrame(rows, columns=['ConversationTranscriptId', 'ConversationStartTime', 'SchemaVersion', 'BotName', 'BotId', 'AADTenantId', 'BatchId', 'SessionStartTimeUtc', 'SessionEndTimeUtc', 'SessionType', 'SessionOutcome', 'TurnCount', 'OutcomeReason', 'ImpliedSuccess', 'LastUserIntentId', 'IsDesignMode', 'Locale', 'UserMessageCount', 'BotMessageCount', 'TotalMessageCount', 'FirstUserMessage', 'LastUserMessage', 'FirstBotMessage', 'LastBotMessage', 'TopicsTriggered', 'TopicCount', 'PrimaryTopic', 'DurationSeconds', 'AverageLatencyMs', 'Var_ESS_UserContext_UPN', 'Var_ESS_Message_Disclaimer', 'Var_ESS_UserContext_RefreshInterval_Hours', 'Var_ESS_UserContext_Conversation_Initialized', 'AllVariables', 'TotalActivityCount', 'TraceActivityCount', 'EventActivityCount', 'StatusCode', 'StateCode', 'FirstErrorCode', 'ErrorCategory', 'IsAuthenticated', 'MultiAgentSession', 'SessionOutcomeExplicit', 'SessionOutcomeReasonExplicit', 'GenerativeResponseCount', 'LLMCallCount', 'PromptTokenCount', 'CompletionTokenCount', 'TotalTokenCount', 'PluginCallCount', 'PluginSuccessCount', 'TopicsStarted', 'TopicsCompleted', 'TopicCompletionRate', 'PrimaryTopicDwellMs', 'TotalTopicDwellMs'])

agent_performance = build_agent_performance(parsed)
print(f'agent performance rows: {len(agent_performance):,}')

## 9. Write Delta tables

First carries `SourceEnvironment` through to the transcript-derived tables (`agent_sessions`, `agent_turns`, `agent_errors`, `agent_subagents`, `agent_performance` get a `source_environment` column via a conversation→environment lookup; `agent_catalogue` gets `source_environments`, the pipe-joined list of environments each agent appears in). Then flattens embedded newlines/tabs in every text cell (one record per physical line — matches the pandas + Power Query behaviour), converts each pandas frame to Spark with an explicit all-string schema, and writes the six Delta tables. The PBIP then sources them from the Lakehouse SQL endpoint (Import) or zero-copy via Direct Lake — the new column appears automatically as an available field on those tables.


In [ ]:
# --- Carry SourceEnvironment through to the transcript-derived tables ---
# `parsed` carries SourceEnvironment per-row (added in cell 2a when TAG_ENVIRONMENT=True).
# The five fact tables key on a conversation id, so map each conversation -> its environment.
_has_env = 'SourceEnvironment' in parsed.columns
if _has_env:
    _conv_env = dict(zip(parsed.get('conversationtranscriptid', pd.Series(dtype=str)),
                         parsed['SourceEnvironment']))
    def _add_env(df, key):
        if df is not None and not df.empty and key in df.columns:
            df = df.copy()
            df['source_environment'] = df[key].map(lambda c: _conv_env.get(c))
        return df
    sessions          = _add_env(sessions,          'conversation_id')
    turns             = _add_env(turns,             'conversation_id')
    errors            = _add_env(errors,            'conversation_id')
    subagents         = _add_env(subagents,         'conversation_id')
    agent_performance = _add_env(agent_performance, 'ConversationTranscriptId')
    # agent_catalogue is a dimension: list the distinct environments each agent appears in.
    if not agent_dim.empty and not sessions.empty and 'source_environment' in sessions.columns:
        _env_by_agent = (sessions.dropna(subset=['primary_agent_schema'])
                         .groupby('primary_agent_schema')['source_environment']
                         .apply(lambda s: '|'.join(sorted({x for x in s if x}))))
        agent_dim = agent_dim.copy()
        agent_dim['source_environments'] = agent_dim['agent_schema'].map(_env_by_agent).fillna('')
    print(f'  SourceEnvironment carried through ({len(set(v for v in _conv_env.values() if v))} environment(s)).')
else:
    print('  No SourceEnvironment column (files mode / untagged) — tables written without it.')

outputs = {
    'agent_sessions':     sessions,
    'agent_turns':        turns,
    'agent_errors':       errors,
    'agent_subagents':    subagents,
    'agent_catalogue':    agent_dim,
    'agent_performance':  agent_performance,
}

_ws = re.compile(r'[\r\n\t]+')
def _flatten(v):
    return _ws.sub(' ', v).strip() if isinstance(v, str) else v

from pyspark.sql.types import StructType, StructField, StringType

# --- Upsert support (WRITE_MODE='merge'): keyed MERGE so re-running an incremental
# load over the SAME environments updates rows in place instead of duplicating them.
MERGE_KEYS = {
    'agent_sessions':     ['conversation_id'],
    'agent_turns':        ['turn_id'],
    'agent_errors':       ['conversation_id', 'error_timestamp_utc', 'error_code'],
    'agent_subagents':    ['conversation_id', 'invocation_timestamp_utc', 'connected_agent_schema', 'plan_step_id'],
    'agent_catalogue':    ['agent_schema'],
    'agent_performance':  ['ConversationTranscriptId'],
}

def _merge_delta(sdf, name, table):
    keys = list(MERGE_KEYS.get(name, []))
    # Multi-env safety: include the environment in the key for the fact tables so the
    # same conversation id from two environments can never collide (catalogue is a dim).
    if 'source_environment' in sdf.columns and name != 'agent_catalogue' and 'source_environment' not in keys:
        keys.append('source_environment')
    from delta.tables import DeltaTable
    if not keys or not spark.catalog.tableExists(table):
        # First load (or no natural key) -> create/extend via append; enables column mapping.
        (sdf.write.mode('append')
            .option('mergeSchema', 'true')
            .option('delta.columnMapping.mode', 'name')
            .option('delta.minReaderVersion', '2')
            .option('delta.minWriterVersion', '5')
            .format('delta').saveAsTable(table))
        return 'append(first-load)'
    cond = ' AND '.join([f't.`{k}` <=> s.`{k}`' for k in keys])
    (DeltaTable.forName(spark, table).alias('t')
        .merge(sdf.alias('s'), cond)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    return f'merge(on {"+".join(keys)})'

_chunk_rows = int(globals().get('PANDAS_TO_SPARK_CHUNK_ROWS', 20000))

for name, pdf in outputs.items():
    pdf = pdf.copy()
    for col in pdf.columns:
        pdf[col] = pdf[col].map(_flatten)
    # All-string write keeps the schema stable across snapshots; the PBIP types on read.
    pdf = pdf.astype(object).where(pd.notna(pdf), None)
    # Force an explicit all-string schema so that columns which are entirely null in a
    # given snapshot (e.g. feedback_verdict when no feedback was submitted) are written as
    # StringType, NOT Spark's void/NullType — otherwise the Lakehouse SQL endpoint won't
    # expose them and the PBIP errors with 'the column ... wasn't found'.
    schema = StructType([StructField(c, StringType(), True) for c in pdf.columns])
    table = f'{OUTPUT_PREFIX}.{name}'
    _pdf_str = pdf.astype(str).where(pd.notna(pdf), None)

    # Write in chunks to avoid hitting spark.rpc.message.maxSize on large tenants.
    if WRITE_MODE == 'merge':
        _how = 'merge(no-rows)'
        _chunks = 0
        for _i in range(0, len(_pdf_str), _chunk_rows):
            _part = _pdf_str.iloc[_i:_i + _chunk_rows]
            _sdf_part = spark.createDataFrame(_part, schema=schema)
            _how = _merge_delta(_sdf_part, name, table)
            _chunks += 1
        print(f'  {table:28} {len(pdf):>7,} rows  ->  {_how}, chunks={_chunks}, chunk_rows={_chunk_rows}')
    else:
        if WRITE_MODE == 'overwrite':
            spark.sql(f'DROP TABLE IF EXISTS {table}')
        _chunks = 0
        if len(_pdf_str) == 0:
            (spark.createDataFrame([], schema=schema)
                .write.mode('append')
                .option('overwriteSchema', 'true')
                .option('delta.columnMapping.mode', 'name')
                .option('delta.minReaderVersion', '2')
                .option('delta.minWriterVersion', '5')
                .format('delta').saveAsTable(table))
            _chunks = 1
        else:
            for _i in range(0, len(_pdf_str), _chunk_rows):
                _part = _pdf_str.iloc[_i:_i + _chunk_rows]
                _mode = 'append' if (WRITE_MODE == 'overwrite' or _chunks > 0) else WRITE_MODE
                (spark.createDataFrame(_part, schema=schema)
                    .write.mode(_mode)
                    .option('overwriteSchema', 'true')
                    .option('delta.columnMapping.mode', 'name')
                    .option('delta.minReaderVersion', '2')
                    .option('delta.minWriterVersion', '5')
                    .format('delta').saveAsTable(table))
                _chunks += 1
        print(f'  {table:28} {len(pdf):>7,} rows  ->  written ({WRITE_MODE}), chunks={_chunks}, chunk_rows={_chunk_rows}')

print('\nDone. Six Delta tables written. Refresh the PBIP / Direct Lake semantic model to pick them up.')
